# 03 · Preparación de los datos

Limpieza, tipado, tratamiento de faltantes, variables derivadas y exportación a `data/processed/`. Cada decisión que modifique los datos originales se justifica aquí.

Este notebook ejecuta, en orden, la lista de acciones de la sección 2.8 del notebook 02 (eliminar, transformar, tratar, crear y conservar). El archivo original de `data/raw/` no se modifica: todo el trabajo se hace sobre una copia en memoria y el resultado se guarda aparte. Al final se comprueba que la base procesada reproduce las cifras de la fase 02 y se exportan dos archivos que alimentan la aplicación de Streamlit.

## 3.1 Importación de librerías y carga de datos

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def miles(v):
    return f"{v:,.0f}".replace(",", ".")

def num(v, decimales=2):
    return f"{v:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")

RUTA_RAW = Path("../data/raw/10_comercio_electronico.csv")
CARPETA_PROCESSED = Path("../data/processed")
CARPETA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
# df_raw queda intacto como referencia; todo el trabajo se hace sobre la copia df
df_raw = pd.read_csv(RUTA_RAW)
df = df_raw.copy()
df.shape

(62000, 19)

## 3.2 Eliminar columnas redundantes

La única columna que se elimina es `entrega_a_tiempo`. Antes de borrarla se vuelve a comprobar que no aporta nada que `dias_retraso` no diga.

In [3]:
# Una orden está a tiempo si y solo si tiene 0 días de retraso
a_tiempo_si = df["entrega_a_tiempo"].eq("Sí")
sin_retraso = df["dias_retraso"].eq(0)
discrepancias = int((a_tiempo_si != sin_retraso).sum())
print("Órdenes donde las dos columnas no coinciden:", discrepancias)
assert discrepancias == 0

Órdenes donde las dos columnas no coinciden: 0


In [4]:
df = df.drop(columns=["entrega_a_tiempo"])
df.shape

(62000, 18)

**Resultado.** La base pasa de **19 a 18 columnas**. El KPI de % de entregas a tiempo se calculará desde `dias_retraso` (órdenes con 0 días de retraso), sin perder ninguna información.

## 3.3 Tipado de las columnas

Se corrigen los tipos que llegaron mal desde el CSV: la fecha (texto), las devoluciones (texto "Sí"/"No"), la calificación (decimal) y las columnas de texto que en realidad son categorías.

In [5]:
# fecha_orden: de texto a fecha
df["fecha_orden"] = pd.to_datetime(df["fecha_orden"], format="%Y-%m-%d")

# devolucion_30d: de "Sí"/"No" a 1/0, para que el % de devoluciones sea un promedio
assert set(df["devolucion_30d"].unique()) == {"Sí", "No"}
df["devolucion_30d"] = df["devolucion_30d"].map({"Sí": 1, "No": 0}).astype("int8")

# calificacion_producto_1a5: de decimal a entero que admite vacíos (Int64)
calificacion = df["calificacion_producto_1a5"]
assert (calificacion.dropna() % 1 == 0).all(), "hay calificaciones con decimales"
df["calificacion_producto_1a5"] = calificacion.astype("Int64")

In [6]:
# Columnas de texto -> categoría, con las categorías ordenadas de más a menos órdenes
columnas_categoricas = ["ciudad_destino", "departamento_destino", "region", "categoria", "tipo_vendedor",
                        "dispositivo", "fuente_trafico", "metodo_pago", "operador_entrega"]
for col in columnas_categoricas:
    orden = df[col].value_counts().index
    df[col] = pd.Categorical(df[col], categories=orden)

df.dtypes.to_frame("tipo")

,tipo
id_orden,str
fecha_orden,datetime64[us]
ciudad_destino,category
departamento_destino,category
region,category
categoria,category
tipo_vendedor,category
dispositivo,category
fuente_trafico,category
valor_bruto_cop,int64


**Resultado.** `fecha_orden` es ahora una fecha, `devolucion_30d` vale 1 o 0, `calificacion_producto_1a5` es entera con vacíos permitidos (`Int64`) y las 9 columnas descriptivas son de tipo categoría, ordenadas de más a menos órdenes, que es el orden en que aparecerán por defecto en tablas y gráficos.

## 3.4 Valores vacíos, atípicos y redondeo

Aquí no se modifica ningún dato: se confirma el estado de los tres puntos que la fase 02 decidió **no** corregir y se dejan documentadas las razones.

In [7]:
# 1) Calificaciones vacías: se dejan vacías
nulos = df.isna().sum()
nulos = nulos[nulos > 0]
print("Columnas con vacíos:", list(nulos.index))
print(f"Órdenes sin calificación: {miles(nulos.iloc[0])} ({num(nulos.iloc[0] / len(df) * 100, 1)} %)")
assert list(nulos.index) == ["calificacion_producto_1a5"] and nulos.iloc[0] == 1364

Columnas con vacíos: ['calificacion_producto_1a5']
Órdenes sin calificación: 1.364 (2,2 %)


In [8]:
# 2) Atípicos por valor neto (regla del rango intercuartílico): se conservan, solo se marcan
q1, q3 = df["valor_neto_cop"].quantile([0.25, 0.75])
LIMITE_ALTO_VALOR = q3 + 1.5 * (q3 - q1)
atipicas = df[df["valor_neto_cop"] > LIMITE_ALTO_VALOR]

print(f"Límite superior: ${miles(LIMITE_ALTO_VALOR)}")
print(f"Órdenes atípicas: {miles(len(atipicas))} ({num(len(atipicas) / len(df) * 100, 1)} %)")
print(f"Participación de Tecnología entre ellas: {num((atipicas['categoria'] == 'Tecnología').mean() * 100, 1)} %")
print(f"Ventas netas que aportan: {num(atipicas['valor_neto_cop'].sum() / df['valor_neto_cop'].sum() * 100, 1)} %")

Límite superior: $598.200
Órdenes atípicas: 5.633 (9,1 %)
Participación de Tecnología entre ellas: 84,8 %
Ventas netas que aportan: 39,0 %


In [9]:
# 3) Redondeo del valor neto: no se corrige
neto_calculado = df["valor_bruto_cop"] * (1 - df["descuento_pct"] / 100)
diferencia = (df["valor_neto_cop"] - neto_calculado).abs()
print(f"Diferencia máxima entre el valor neto de la base y el recalculado: ${num(diferencia.max())}")

Diferencia máxima entre el valor neto de la base y el recalculado: $319,50


**Decisiones.**

- **Calificaciones vacías.** Las **1.364 órdenes sin calificación (2,2 %)** se dejan vacías y se excluyen del promedio. Están repartidas al azar entre categorías, devoluciones y entregas, así que rellenarlas no aportaría información y sí podría mover el KPI de calificación promedio.
- **Atípicos.** Las **5.633 órdenes** (9,1 %) por encima del límite del rango intercuartílico se **conservan**: son compras legítimas, en su mayoría de Tecnología (84,8 %), y aportan cerca del 39 % de las ventas netas. Eliminarlas distorsionaría el KPI de ventas netas. En la sección 3.6 se marcan con un indicador para poder filtrarlas o compararlas sin borrarlas.
- **Redondeo.** `valor_neto_cop` se usa **tal como viene**, porque es lo que el cliente realmente pagó. Las diferencias frente al recalculo son de pocos pesos (menos del 0,02 %) y se explican por el redondeo del descuento a dos decimales.

## 3.5 Tabla calendario

Se crea una tabla con un día por fila que cubre todo el periodo de los datos, del 1 de enero de 2024 al 30 de septiembre de 2026. Los nombres de meses y días se escriben a mano para que no dependan del idioma del equipo donde se ejecute el notebook.

In [10]:
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre",
         "Octubre", "Noviembre", "Diciembre"]
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

calendario = pd.DataFrame({"fecha": pd.date_range("2024-01-01", "2026-09-30", freq="D")})
calendario["anio"] = calendario["fecha"].dt.year
calendario["trimestre"] = calendario["fecha"].dt.quarter
calendario["mes"] = calendario["fecha"].dt.month
calendario["nombre_mes"] = pd.Categorical(calendario["mes"].map(lambda m: MESES[m - 1]),
                                          categories=MESES, ordered=True)
calendario["anio_mes"] = calendario["fecha"].dt.to_period("M").dt.to_timestamp()   # primer día del mes
calendario["dia_semana_num"] = calendario["fecha"].dt.dayofweek                      # 0 = lunes
calendario["dia_semana"] = pd.Categorical(calendario["dia_semana_num"].map(lambda d: DIAS[d]),
                                          categories=DIAS, ordered=True)
# Enero a septiembre existe en los tres años; octubre a diciembre solo en 2024 y 2025
calendario["periodo_comparable"] = calendario["mes"] <= 9

print("Días en la tabla calendario:", len(calendario))
assert calendario["fecha"].is_unique
assert calendario["fecha"].min() == df["fecha_orden"].min() and calendario["fecha"].max() == df["fecha_orden"].max()
assert df["fecha_orden"].isin(calendario["fecha"]).all(), "hay órdenes con fecha fuera del calendario"
calendario.head()

Días en la tabla calendario: 1004


,fecha,anio,trimestre,mes,nombre_mes,anio_mes,dia_semana_num,dia_semana,periodo_comparable
0,2024-01-01,2024,1,1,Enero,2024-01-01,0,Lunes,True
1,2024-01-02,2024,1,1,Enero,2024-01-01,1,Martes,True
2,2024-01-03,2024,1,1,Enero,2024-01-01,2,Miércoles,True
3,2024-01-04,2024,1,1,Enero,2024-01-01,3,Jueves,True
4,2024-01-05,2024,1,1,Enero,2024-01-01,4,Viernes,True


In [11]:
# Los atributos del calendario se unen a las órdenes por fecha (muchas órdenes -> un día)
df = df.merge(calendario, how="left", left_on="fecha_orden", right_on="fecha",
              validate="many_to_one").drop(columns="fecha")
assert df["anio"].notna().all()
df.shape

(62000, 26)

**Resultado.** La tabla calendario tiene **1.004 días** y todas las órdenes encuentran su fecha en ella (el calendario no deja fechas fuera ni inventa meses). Contiene año, trimestre, mes, nombre del mes, primer día del mes (`anio_mes`, para las series de tiempo), día de la semana y el indicador `periodo_comparable`, que vale verdadero de enero a septiembre y permite comparar 2024, 2025 y 2026 sobre los mismos meses.

## 3.6 Variables derivadas

Se crean las columnas que el dashboard necesita para responder los objetivos: cuánto dinero se deja de recibir por descuentos, los rangos que muestran el efecto del descuento y del retraso, el tipo de promesa de entrega y el indicador de orden de alto valor.

In [12]:
# Cuánto ingreso se deja de recibir por descuento, en pesos
df["valor_descuento_cop"] = df["valor_bruto_cop"] - df["valor_neto_cop"]

# Rango de descuento (el límite izquierdo se incluye: 5 % cae en "5–10 %")
df["rango_descuento"] = pd.cut(df["descuento_pct"], bins=[0, 5, 10, 15, 20, 100], right=False,
                               labels=["0–5 %", "5–10 %", "10–15 %", "15–20 %", "20 % o más"], ordered=True)

# Rango de retraso
df["rango_retraso"] = pd.cut(df["dias_retraso"], bins=[-1, 0, 2, 4],
                             labels=["Sin retraso", "1–2 días", "3–4 días"], ordered=True)

# Tipo de promesa de entrega: aísla la entrega el mismo día (0 días prometidos)
df["tipo_promesa"] = pd.cut(df["dias_prometidos"], bins=[-1, 0, 2, np.inf],
                            labels=["Mismo día", "1–2 días", "3 o más días"], ordered=True)

# Orden de alto valor: atípica por valor neto, conservada y solo marcada
df["alto_valor"] = df["valor_neto_cop"] > LIMITE_ALTO_VALOR

In [13]:
# Cuántas órdenes caen en cada rango
num_1 = lambda v: num(v, 1)
for col in ["rango_descuento", "rango_retraso", "tipo_promesa"]:
    print(f"\n{col}")
    print(df[col].value_counts(sort=False).to_frame("órdenes").assign(**{"%": lambda t: (t["órdenes"] / len(df) * 100).round(1).map(num_1)}).to_string())
print("\nalto_valor:", int(df["alto_valor"].sum()), "órdenes")


rango_descuento
                 órdenes     %
rango_descuento               
0–5 %              19117  30,8
5–10 %             14927  24,1
10–15 %            13672  22,1
15–20 %             8998  14,5
20 % o más          5286   8,5

rango_retraso
               órdenes     %
rango_retraso               
Sin retraso      54604  88,1
1–2 días          3683   5,9
3–4 días          3713   6,0

tipo_promesa
              órdenes     %
tipo_promesa               
Mismo día        2401   3,9
1–2 días        30960  49,9
3 o más días    28639  46,2

alto_valor: 5633 órdenes


**Resultado.** Se agregan 5 columnas derivadas (`valor_descuento_cop`, `rango_descuento`, `rango_retraso`, `tipo_promesa` y `alto_valor`) además de las 8 del calendario. Los tres rangos son categorías **ordenadas**, de modo que los gráficos las muestren de menor a mayor y no alfabéticamente. Ninguna orden queda sin rango.

## 3.7 Verificación contra la fase 02

Antes de exportar se comprueba que la base procesada no perdió ni cambió nada de lo que la fase 02 había medido: mismas filas, mismos vacíos y las mismas cifras clave de cada hallazgo.

In [14]:
ventas = df["valor_neto_cop"]
por_rango = df.groupby("rango_descuento", observed=True)["valor_neto_cop"].mean()
tec = df["categoria"] == "Tecnología"

verificaciones = [
    ("Órdenes (filas)",                              62000,  len(df)),
    ("Órdenes únicas (id_orden)",                    62000,  df["id_orden"].nunique()),
    ("Calificaciones vacías",                        1364,   int(df["calificacion_producto_1a5"].isna().sum())),
    ("Órdenes de alto valor",                        5633,   int(df["alto_valor"].sum())),
    ("Órdenes sin retraso",                          54604,  int((df["dias_retraso"] == 0).sum())),
    ("Tecnología: % de las órdenes",                 21.2,   round(tec.mean() * 100, 1)),
    ("Tecnología: % de las ventas netas",            52.2,   round(ventas[tec].sum() / ventas.sum() * 100, 1)),
    ("Ticket promedio con descuento < 5 % (COP)",    266852, round(por_rango["0–5 %"])),
    ("Ticket promedio con descuento ≥ 20 % (COP)",   214045, round(por_rango["20 % o más"])),
    ("Calificación promedio sin retraso",            4.31,   round(df.loc[df["dias_retraso"] == 0, "calificacion_producto_1a5"].mean(), 2)),
    ("Calificación promedio con 4 días de retraso",  3.89,   round(df.loc[df["dias_retraso"] == 4, "calificacion_producto_1a5"].mean(), 2)),
    ("% devoluciones con promesa el mismo día",      8.20,   round(df.loc[df["tipo_promesa"] == "Mismo día", "devolucion_30d"].mean() * 100, 2)),
]
tabla = pd.DataFrame(verificaciones, columns=["Verificación", "Esperado (notebook 02)", "Obtenido"])
tabla["Coincide"] = np.isclose(tabla["Esperado (notebook 02)"].astype(float), tabla["Obtenido"].astype(float))
print("Todas coinciden:", bool(tabla["Coincide"].all()))
assert tabla["Coincide"].all()

def formato(v):
    return miles(v) if float(v).is_integer() else num(v)

tabla_vista = tabla.copy()
for col in ["Esperado (notebook 02)", "Obtenido"]:
    tabla_vista[col] = tabla_vista[col].map(formato)
tabla_vista

Todas coinciden: True


,Verificación,Esperado (notebook 02),Obtenido,Coincide
0,Órdenes (filas),62.000,62.000,True
1,Órdenes únicas (id_orden),62.000,62.000,True
2,Calificaciones vacías,1.364,1.364,True
3,Órdenes de alto valor,5.633,5.633,True
4,Órdenes sin retraso,54.604,54.604,True
5,Tecnología: % de las órdenes,"21,20","21,20",True
6,Tecnología: % de las ventas netas,"52,20","52,20",True
7,Ticket promedio con descuento < 5 % (COP),266.852,266.852,True
8,Ticket promedio con descuento ≥ 20 % (COP),214.045,214.045,True
9,Calificación promedio sin retraso,"4,31","4,31",True


**Resultado.** La base procesada reproduce todas las cifras de la fase 02, así que las transformaciones no alteraron el contenido de los datos: solo su forma.

## 3.8 Definición de los KPI

El notebook 01 dejó pendiente el cálculo exacto de cada indicador "hasta conocer el contenido del conjunto de datos". Con la base procesada, así se calcula cada uno. Esta es la definición que usará el dashboard.

| KPI | Cálculo sobre la base procesada |
|---|---|
| Ventas netas | Suma de `valor_neto_cop`. |
| Ticket promedio | Promedio de `valor_neto_cop`; la mediana se muestra como apoyo porque las compras de alto valor suben el promedio. |
| Descuento promedio | Promedio de `descuento_pct`. |
| % entrega a tiempo | Órdenes con `dias_retraso` igual a 0, sobre el total de órdenes. |
| % devoluciones | Promedio de `devolucion_30d` (1 = devuelta) por 100. |
| Calificación promedio | Promedio de `calificacion_producto_1a5`, ignorando las órdenes sin calificación. |

In [15]:
kpis = pd.Series({
    "Ventas netas (millones de COP)": ventas.sum() / 1e6,
    "Ticket promedio (COP)":          ventas.mean(),
    "Ticket mediano (COP)":           ventas.median(),
    "Descuento promedio (%)":         df["descuento_pct"].mean(),
    "% entrega a tiempo":             (df["dias_retraso"] == 0).mean() * 100,
    "% devoluciones":                 df["devolucion_30d"].mean() * 100,
    "Calificación promedio (1 a 5)":  df["calificacion_producto_1a5"].mean(),
}, name="Total")
kpis.map(lambda v: num(v, 1) if v > 1000 else num(v)).to_frame()

,Total
Ventas netas (millones de COP),"15.358,4"
Ticket promedio (COP),"247.716,1"
Ticket mediano (COP),"138.391,5"
Descuento promedio (%),"9,55"
% entrega a tiempo,"88,07"
% devoluciones,"6,66"
Calificación promedio (1 a 5),"4,27"


## 3.9 Exportación

Se guardan dos archivos en `data/processed/`:

- `ordenes_limpias.csv`: una fila por orden, ya con los atributos del calendario y las variables derivadas. Es el archivo que lee la aplicación de Streamlit.
- `calendario.csv`: la tabla calendario por sí sola, para documentar el modelo de datos.

**Modelo de datos.** Streamlit no tiene un motor de relaciones como Power BI, así que la base se entrega ya integrada: una tabla de hechos con las órdenes (una fila por orden, medidas y dimensiones) a la que se le unieron los atributos de la dimensión calendario por fecha. La relación original es de muchas órdenes a un día. La jerarquía geográfica (región → departamento → ciudad) se conserva como columnas de la misma tabla.

**Importante al leer el CSV.** El CSV no guarda tipos, así que al cargarlo en la aplicación hay que volver a indicar las fechas y la calificación entera, como se muestra en la verificación de abajo.

In [16]:
columnas_finales = [
    "id_orden",
    "fecha_orden", "anio", "trimestre", "mes", "nombre_mes", "anio_mes", "dia_semana_num", "dia_semana",
    "periodo_comparable",
    "ciudad_destino", "departamento_destino", "region",
    "categoria", "tipo_vendedor", "dispositivo", "fuente_trafico", "metodo_pago", "operador_entrega",
    "valor_bruto_cop", "descuento_pct", "valor_descuento_cop", "rango_descuento", "valor_neto_cop", "alto_valor",
    "dias_prometidos", "tipo_promesa", "dias_retraso", "rango_retraso",
    "devolucion_30d", "calificacion_producto_1a5",
]
assert sorted(columnas_finales) == sorted(df.columns), "las columnas finales no coinciden con las del DataFrame"
df = df[columnas_finales]

df.to_csv(CARPETA_PROCESSED / "ordenes_limpias.csv", index=False, encoding="utf-8")
calendario.to_csv(CARPETA_PROCESSED / "calendario.csv", index=False, encoding="utf-8")
print("Guardados:", sorted(p.name for p in CARPETA_PROCESSED.glob("*.csv")))

Guardados: ['calendario.csv', 'ordenes_limpias.csv']


In [17]:
# Lectura de prueba, tal como la haría la aplicación de Streamlit
leido = pd.read_csv(CARPETA_PROCESSED / "ordenes_limpias.csv",
                    parse_dates=["fecha_orden", "anio_mes"],
                    dtype={"calificacion_producto_1a5": "Int64"})
assert leido.shape == df.shape
assert leido["valor_neto_cop"].sum() == df["valor_neto_cop"].sum()
assert leido["calificacion_producto_1a5"].isna().sum() == 1364
print("Filas y columnas:", leido.shape)
leido.dtypes.to_frame("tipo").T

Filas y columnas: (62000, 31)


,id_orden,fecha_orden,anio,trimestre,mes,nombre_mes,anio_mes,dia_semana_num,dia_semana,periodo_comparable,ciudad_destino,departamento_destino,region,categoria,tipo_vendedor,dispositivo,fuente_trafico,metodo_pago,operador_entrega,valor_bruto_cop,descuento_pct,valor_descuento_cop,rango_descuento,valor_neto_cop,alto_valor,dias_prometidos,tipo_promesa,dias_retraso,rango_retraso,devolucion_30d,calificacion_producto_1a5
tipo,str,datetime64[us],int64,int64,int64,str,datetime64[us],int64,str,bool,str,str,str,str,str,str,str,str,str,int64,float64,int64,str,int64,bool,int64,str,int64,str,int64,Int64
